# 05. mergeInto - 行ごとに「どうするか」を指示する

`04` では `replaceUsing` を扱いました。キー列が一致する行を **丸ごと置き換える** 仕組みです。

ただ実務では、もう少し細かい指示を出したくなります。

- 金額だけ更新したい。他の列は触りたくない
- 既にある行は更新、無い行は追加、という処理を1回でやりたい
- 上流から消えた行は、こちらでも消したい
- ある条件のときだけ更新したい

これらを行ごとに指示できるのが `mergeInto` です。

このノートブックで確かめること:

1. `mergeInto` の基本形。一致したら更新、しなかったら追加
2. **列を選んで**更新できること (`replaceUsing` との違い)
3. `whenNotMatchedBySource` で、上流から消えた行を扱えること
4. 条件を付けて、更新するかどうかを選べること
5. **ソースにキーの重複があるとどうなるか**

**前提**: `00_setup` を実行済みであること。`04` を読んでいること。

## 準備

In [ ]:
from databricks.connect import DatabricksSession
from pyspark.sql.functions import col, lit

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [ ]:
CATALOG = "tech_survey"
TABLE = f"{CATALOG}.silver.merge_orders"

# ターゲット側の列を指すときに使う。完全修飾名ではなく、末尾のテーブル名で参照する
TABLE_SHORT = "merge_orders"

## 1. テーブルを用意する

`04` と同じく Liquid Clustering で作ります。
今回は `status` 列を足しました。「一部の列だけ更新する」を試すためです。

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

# DDL: クラスタリングキーは order_date にする
spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id INT,
        product STRING,
        amount INT,
        status STRING,
        order_date DATE
    )
    CLUSTER BY (order_date)
""")

# DML: 初期データを3件入れる
spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        (1, 'laptop',   150000, 'placed', DATE '2026-09-11'),
        (2, 'monitor',   40000, 'placed', DATE '2026-09-11'),
        (3, 'keyboard',  12000, 'placed', DATE '2026-09-11')
""")

# 出発点の状態を確認する
display(spark.table(TABLE).orderBy("order_id"))

## 2. `mergeInto` の基本形

「2つの表を突き合わせて、行ごとに処理を決める」命令です。組み立てはこうなります。

```python
(
    ソース.alias("s")                      # 変更する材料
    .mergeInto(ターゲット, 突き合わせ条件)   # 変更される側と、対応の取り方
    .whenMatched().～                       # 両方にあった行をどうするか
    .whenNotMatched().～                    # ソースにだけあった行をどうするか
    .merge()                                # 実行する
)
```

**`.merge()` を呼ぶまで何も起きません。** チェーンの最後を落とすと、
エラーも出ないのにテーブルが変わらない、という分かりにくい状態になります。

突き合わせ条件では、ソース側を `s.列名`、ターゲット側を **`テーブル名.列名`** で指します。
ターゲットは完全修飾名 (`カタログ.スキーマ.テーブル`) ではなく、末尾のテーブル名だけを使います。

まず「あれば更新、なければ追加」をやります。
ソースには **既存の order_id=1 と、存在しない order_id=4** を混ぜてあります。

In [ ]:
# 更新1件 (order_id=1) と 新規1件 (order_id=4) を含むソース
source = spark.createDataFrame(
    [
        (1, "laptop", 155000, "shipped", "2026-09-11"),
        (4, "mouse", 5000, "placed", "2026-09-11"),
    ],
    "order_id INT, product STRING, amount INT, status STRING, order_date DATE",
)

# これから何を書き込むのかを先に見ておく
display(source)

In [ ]:
(
    source.alias("s")
    .mergeInto(TABLE, col(f"{TABLE_SHORT}.order_id") == col("s.order_id"))  # order_id で突き合わせる
    .whenMatched()  # 両方にある行
    .updateAll()  # 全列をソースの値で上書きする
    .whenNotMatched()  # ソースにしかない行
    .insertAll()  # そのまま追加する
    .merge()  # ここで実行される
)

# order_id=1 が更新され、order_id=4 が増えていることを確認する
display(spark.table(TABLE).orderBy("order_id"))

`updateAll()` と `insertAll()` は「全列をまとめて」という指定です。
ここまでは `replaceUsing` でもほぼ同じことができます。

## 3. 列を選んで更新する

ここからが `mergeInto` の本領です。

「金額だけ直したい。ステータスは今の値を保ちたい」という状況を考えます。
`replaceUsing` は行を丸ごと置き換えるので、この指示は出せません。
`mergeInto` なら `update()` に **更新したい列だけ** を渡せます。

order_id=1 は今 `shipped` になっています。金額だけ変えて、これが残るかを見ます。

In [ ]:
# 突き合わせに使う order_id と、更新したい amount だけを持つソース
amount_fix = spark.createDataFrame(
    [(1, 160000)],
    "order_id INT, amount INT",
)

display(amount_fix)

In [ ]:
(
    amount_fix.alias("s")
    .mergeInto(TABLE, col(f"{TABLE_SHORT}.order_id") == col("s.order_id"))
    .whenMatched()
    .update({"amount": col("s.amount")})  # amount だけ更新する。status や product には触らない
    .merge()
)

# order_id=1 の amount だけが変わり、status が shipped のままであることを確認する
display(spark.table(TABLE).orderBy("order_id"))

ソースに `amount` しか持たせていない点にも注目してください。
突き合わせに使う `order_id` と、更新したい列さえあれば動きます。全列を揃える必要がありません。

`update()` に渡すのは `{列名: 値}` の辞書です。値は式なので、
`col("s.amount") * 1.1` のような計算も書けます。

## 4. 上流から消えた行をどうするか

もう1つ、`mergeInto` にしかできないことがあります。

**ソースに無かったターゲットの行** を処理する `whenNotMatchedBySource` です。

状況を考えます。上流が「9月11日の注文はこの2件が全てです」と言ってきたとします。
ターゲットには4件あるので、残り2件は取り消されたことになります。

| メソッド | 対象 |
|---|---|
| `whenMatched` | 両方にある行 |
| `whenNotMatched` | **ソースにだけ**ある行 |
| `whenNotMatchedBySource` | **ターゲットにだけ**ある行 |

実行前に、4件のうちどれが残るか予想してください。

In [ ]:
# 上流が「これが全て」と言ってきた2件
latest = spark.createDataFrame(
    [
        (1, "laptop", 160000, "shipped", "2026-09-11"),
        (2, "monitor", 40000, "placed", "2026-09-11"),
    ],
    "order_id INT, product STRING, amount INT, status STRING, order_date DATE",
)

display(latest)

In [ ]:
(
    latest.alias("s")
    .mergeInto(TABLE, col(f"{TABLE_SHORT}.order_id") == col("s.order_id"))
    .whenMatched()
    .updateAll()
    .whenNotMatched()
    .insertAll()
    .whenNotMatchedBySource()  # ソースに無かった行
    .delete()  # 上流で取り消されたとみなして消す
    .merge()
)

# ソースにあった2件だけが残っていることを確認する
display(spark.table(TABLE).orderBy("order_id"))

`04` で見た `replaceWhere` も「範囲ごと入れ替える」ので似た結果になりますが、考え方が違います。

- `replaceWhere` … **範囲を条件で宣言**して、その中身を丸ごと差し替える
- `whenNotMatchedBySource` … **行ごとに突き合わせて**、余った行を処理する

後者は「消す」以外も選べます。`.delete()` の代わりに
`.update({"status": lit("cancelled")})` と書けば、消さずに印を付けられます。
履歴を残したい場合はこちらになります。

## 5. 条件を付ける

`whenMatched()` には条件を渡せます。「一致していて、かつ〜のときだけ」という指定です。

よくあるのは、**古い情報で新しい情報を上書きしないようにする** 使い方です。
順序が前後して届いたデータで、せっかくの更新を巻き戻してしまう事故を防げます。

ここでは「すでに `shipped` になっている注文は、`placed` に戻さない」を表現します。

In [ ]:
# order_id=1 を placed に戻そうとする指示が混ざったソース
status_update = spark.createDataFrame(
    [
        (1, "placed"),
        (2, "shipped"),
    ],
    "order_id INT, status STRING",
)

display(status_update)

In [ ]:
(
    status_update.alias("s")
    .mergeInto(TABLE, col(f"{TABLE_SHORT}.order_id") == col("s.order_id"))
    .whenMatched(col(f"{TABLE_SHORT}.status") != lit("shipped"))  # ターゲットが shipped のときは対象外
    .update({"status": col("s.status")})
    .merge()
)

# order_id=1 は shipped のまま、order_id=2 だけが shipped に変わることを確認する
display(spark.table(TABLE).orderBy("order_id"))

## 6. ソースにキーの重複があると

`mergeInto` を使う上で最も引っかかりやすい点です。

ソース側に同じキーの行が2つあると、ターゲットの1行に対して2つの更新候補ができます。
Deltaはどちらを採用すべきか決められません。どうなるか試します。

In [ ]:
# order_id=1 が2行ある。どちらの金額を採用すべきか決められない
duplicated = spark.createDataFrame(
    [
        (1, 170000),
        (1, 180000),
    ],
    "order_id INT, amount INT",
)

display(duplicated)

In [ ]:
# エラーメッセージを読みたいので、ここでは例外を捕まえて表示する
try:
    (
        duplicated.alias("s")
        .mergeInto(TABLE, col(f"{TABLE_SHORT}.order_id") == col("s.order_id"))
        .whenMatched()
        .update({"amount": col("s.amount")})
        .merge()
    )
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:400])

エラーになったはずです。

黙ってどちらかが採用されるより、はるかに親切な動作です。
「なぜか金額が実行するたびに変わる」という追いにくい不具合になるところを、実行時点で止めてくれます。

実務ではソースに重複が入り込むことがよくあります。
上流が同じレコードを2回送ってきた、結合で行が増えた、などです。
`mergeInto` の前に **キーで重複を排除しておく** のが定石になります。
「どれを残すか」(最新のタイムスタンプのものを残す、など) は自分で決める必要があります。

## 7. `replaceUsing` との使い分け

| | `replaceUsing` (04) | `mergeInto` (05) |
|---|---|---|
| 更新の単位 | 行を丸ごと置き換える | **列を選んで**更新できる |
| ソースに必要な列 | 全列を揃える | キーと更新する列だけでよい |
| ターゲットにだけある行 | 触らない | 削除も更新もできる |
| 条件付きの更新 | できない | `whenMatched(条件)` で指定できる |
| ソースのキー重複 | エラーにならない | **エラーになる** |

できることは `mergeInto` のほうが多いですが、その分「何が起きるか」を自分で決める必要があります。

「届いた行で丸ごと置き換えればよい」だけなら `replaceUsing` のほうが短く書け、意図も明確になります。
列単位の制御や削除の伝播が要るときに `mergeInto` を選ぶ、という順序で考えるとよさそうです。

## 8. SQLとの対応

ここまで PySpark で書いてきましたが、ドキュメントやネット上の記事は
SQLの `MERGE INTO` で書かれているものが大半です。読めるように対応を控えておきます。

| PySpark | SQL |
|---|---|
| `.mergeInto(表, 条件)` | `MERGE INTO 表 AS t USING ソース AS s ON 条件` |
| `.whenMatched().updateAll()` | `WHEN MATCHED THEN UPDATE SET *` |
| `.whenMatched().update({"a": col("s.a")})` | `WHEN MATCHED THEN UPDATE SET t.a = s.a` |
| `.whenMatched(条件).update(...)` | `WHEN MATCHED AND 条件 THEN UPDATE SET ...` |
| `.whenMatched().delete()` | `WHEN MATCHED THEN DELETE` |
| `.whenNotMatched().insertAll()` | `WHEN NOT MATCHED THEN INSERT *` |
| `.whenNotMatchedBySource().delete()` | `WHEN NOT MATCHED BY SOURCE THEN DELETE` |
| `.merge()` | (SQLは文の実行で完了する) |

句の構造は同じなので、片方が読めればもう片方も読めます。

なお、`delta.tables.DeltaTable.merge()` という書き方もあります。
Spark 4.0 で `DataFrame.mergeInto` が入る前から使われてきたAPIで、古い記事ではこちらが出てきます。

## 考えてみる

- `3.` でソースに `order_id` と `amount` しか持たせませんでした。これが `replaceUsing` ではできないのはなぜでしょうか
- `4.` で `.delete()` ではなく `.update({"status": lit("cancelled")})` にすると、何が嬉しいでしょうか
- `6.` の重複を排除するとき、「どれを残すか」はどう決めればよいでしょうか

### 答え

**Q1. なぜ `replaceUsing` では列を選べないのか**

`replaceUsing` は **行の置き換え** だからです。
キーが一致した行を消して、ソースの行をそのまま入れる、という動作をします。
ソースに `status` が無ければ、置き換わった後の行にも `status` はありません。

`mergeInto` の `update()` は **既存の行を書き換える** 操作です。
指定しなかった列は元の値のまま残ります。同じ「更新」でも、成り立ちが違います。

**Q2. 消さずに印を付ける利点**

3つあります。

- **後から追える**。いつ取り消されたのか、何が取り消されたのかが残る
- **下流に伝わる**。行が消えるだけだと、下流は「無くなったこと」に気づきにくい。
  `cancelled` という行が来れば、`03` で見たCDFの変更として拾えます
- **戻せる**。取り消しが誤りだった場合に、status を戻すだけで復旧できる

一方で、行が増え続けるので、いつか整理する仕組みは要ります。

**Q3. 重複のうちどれを残すか**

データの性質によるので、決まった正解はありません。よく使うのは次の考え方です。

- **更新時刻の列があるなら、最も新しいもの**。これが最も素直です
- 時刻が無いなら、**上流のシーケンス番号やファイルの到着順**
- どちらも無い場合は、そもそも「どれが正しいか」を決められません。
  上流に順序が分かる情報を足してもらうのが本筋になります

「とりあえず1件に絞る」とだけ決めて適当に選ぶと、実行するたびに結果が変わる
不安定な処理になります。`06_idempotent_writes` で扱う「何度実行しても同じ結果になる」
という性質が、ここで壊れます。

## 後片付け

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")